## Import necessary libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
from eda_toolkit import ensure_directory, generate_table1

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import gaussian_kde

sys.path.append("../")

## Paths

In [ ]:
data_path = "../data/raw"

In [ ]:
base_path = os.path.join(os.pardir)

# create image paths
image_path_png = os.path.join(base_path, "images", "png_images")
image_path_svg = os.path.join(base_path, "images", "svg_images")

# Use the function to ensure'data' directory exists
ensure_directory(image_path_png)
ensure_directory(image_path_svg)

In [ ]:
df = pd.read_csv(os.path.join(data_path, "ACLED Data_2026-01-02.csv")).set_index(
    "event_id_cnty"
)

In [ ]:
df.head()

In [ ]:
df["disorder_type"].value_counts()

In [ ]:
df["fatality_yes_no"] = np.where(df["fatalities"] > 0, "Yes", "No")

In [ ]:
df["fatality_yes_no"]

In [ ]:
df_table1 = df[
    [
        "sub_event_type",
        "fatality_yes_no",
    ]
]

In [ ]:
df_table1["sub_event_type"].value_counts()

In [ ]:
table_1 = generate_table1(
    df_table1,
    value_counts=True,
    groupby_col="fatality_yes_no",
    drop_variables="fatality_yes_no",
)

In [ ]:
table_1 = table_1.drop(columns=["Type", "Mode", "Missing (n)", "Missing (%)"])

In [ ]:
df["civilian_targeting"].value_counts()

In [ ]:
table_1

## Fatality Distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import gaussian_kde

fig = plt.figure(figsize=(14, 5))
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

zero_pct = (df["fatalities"] == 0).mean() * 100
log_all = np.log1p(df["fatalities"])
log_fatal = np.log1p(df[df["fatalities"] > 0]["fatalities"])

# ── Panel 1: log-transformed, ALL events ──────────────────────────────────
ax1 = fig.add_subplot(gs[0])

ax1.hist(
    log_all,
    bins=50,
    color="steelblue",
    edgecolor="white",
    linewidth=0.5,
    density=True,
    alpha=0.75,
)

ax1.axvline(
    log_all.mean(),
    color="blue",
    linestyle="--",
    linewidth=2,
    label=f"Mean = {log_all.mean():.2f}",
)
ax1.axvline(
    df["fatalities"].median(),
    color="crimson",
    linestyle="--",
    linewidth=2,
    label=f"Median = {int(df['fatalities'].median())}",
)


ax1.set_xlabel("log(fatalities + 1)", fontsize=13)
ax1.set_ylabel("Density", fontsize=13)
ax1.set_title(f"All events (n = {len(df):,})", fontsize=12)
ax1.legend(fontsize=12, framealpha=0.9)
ax1.tick_params(labelsize=11)

# ── Panel 2: log-transformed, FATAL events only ───────────────────────────
ax2 = fig.add_subplot(gs[1])

ax2.hist(
    log_fatal,
    bins=40,
    color="steelblue",
    edgecolor="white",
    linewidth=0.5,
    density=True,
    alpha=0.6,
)

kde = gaussian_kde(log_fatal)
x_range = np.linspace(log_fatal.min(), log_fatal.max(), 300)
ax2.plot(x_range, kde(x_range), color="red", linewidth=2.5, label="KDE")

mu, sigma = log_fatal.mean(), log_fatal.std()
ax2.axvline(mu, color="blue", linestyle="--", linewidth=2, label=f"Mean = {mu:.2f}")
ax2.axvline(
    log_fatal.median(),
    color="crimson",
    linestyle="--",
    linewidth=2,
    label=f"Median = {log_fatal.median():.2f}",
)
ax2.axvspan(mu - sigma, mu + sigma, alpha=0.08, color="purple", label="±1 SD")
ax2.axvspan(mu - 2 * sigma, mu + 2 * sigma, alpha=0.05, color="green", label="±2 SD")

ax2.set_xlim(left=0)
ax2.set_xlabel("log(fatalities + 1)", fontsize=13)
ax2.set_ylabel("Density", fontsize=13)
ax2.set_title(f"Fatal events only (n = {len(log_fatal):,})", fontsize=12)
ax2.legend(fontsize=11, framealpha=0.9)
ax2.tick_params(labelsize=11)

# fig.suptitle(
#     f"Fatality distribution — {zero_pct:.1f}% of events report zero fatalities",
#     fontsize=14,
#     fontweight="bold",
#     y=1.02,
# )

plt.savefig(
    os.path.join(image_path_png, "fatality_distribution.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    os.path.join(image_path_svg, "fatality_distribution.svg"),
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.5)

# ── Panel 1: Mean fatalities by interaction type ───────────────────────────
ax1 = fig.add_subplot(gs[0])

interaction_means = (
    df.groupby("interaction")["fatalities"].mean().sort_values(ascending=True)
)

bars1 = ax1.barh(
    interaction_means.index.astype(str),
    interaction_means.values,
    color="steelblue",
    edgecolor="white",
    linewidth=0.5,
    height=0.65,
)

# Value labels
for bar, val in zip(bars1, interaction_means.values):
    ax1.text(
        val + 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.2f}",
        va="center",
        ha="left",
        fontsize=10,
        color="dimgray",
    )

ax1.set_xlabel("Mean fatalities per event", fontsize=12)
ax1.set_ylabel("Interaction code", fontsize=12)
ax1.set_title("Mean fatalities by interaction type", fontsize=13, fontweight="bold")
ax1.tick_params(labelsize=10)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.set_xlim(0, interaction_means.max() * 1.2)

# ── Panel 2: Mean fatalities by sub_event_type ────────────────────────────
ax2 = fig.add_subplot(gs[1])

subtype_means = (
    df.groupby("sub_event_type")["fatalities"].mean().sort_values(ascending=True)
)

# Color-code: top 3 in coral, rest in steelblue
colors = [
    "#d85a30" if v >= subtype_means.nlargest(3).min() else "steelblue"
    for v in subtype_means.values
]

bars2 = ax2.barh(
    subtype_means.index,
    subtype_means.values,
    color=colors,
    edgecolor="white",
    linewidth=0.5,
    height=0.65,
)

# Value labels
for bar, val in zip(bars2, subtype_means.values):
    ax2.text(
        val + 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.2f}",
        va="center",
        ha="left",
        fontsize=9,
        color="dimgray",
    )

ax2.set_xlabel("Mean fatalities per event", fontsize=12)
ax2.set_ylabel("Sub-event type", fontsize=12)
ax2.set_title("Mean fatalities by sub-event type", fontsize=13, fontweight="bold")
ax2.tick_params(labelsize=9)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.set_xlim(0, subtype_means.max() * 1.2)

# Top 3 legend
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor="#d85a30", label="Top 3 by mean fatalities"),
    Patch(facecolor="steelblue", label="All others"),
]
ax2.legend(handles=legend_elements, fontsize=10, framealpha=0.9, loc="lower right")

# fig.suptitle(
#     "Mean fatalities per event by interaction type and sub-event type",
#     fontsize=14,
#     fontweight="bold",
#     y=1.02,
# )

plt.savefig(
    os.path.join(image_path_png, "mean_fatalities_by_feature.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    os.path.join(image_path_svg, "mean_fatalities_by_feature.svg"),
    dpi=300,
    bbox_inches="tight",
)

plt.show()